# Data Quality Assessment for IEEE-CIS Fraud Detection

This notebook performs an initial data quality assessment on the train transaction and identity datasets, including missing values, duplicate identifiers, identity coverage, and basic anomaly checks.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("../../data/raw")
FIGURES_DIR = Path("../../reports/figures")
SUMMARY_DIR = Path("../../reports/model_reports")

TRAIN_TRANSACTION_FILE = DATA_DIR / "train_transaction.csv"
TRAIN_IDENTITY_FILE = DATA_DIR / "train_identity.csv"

QUALITY_SUMMARY_FILE = SUMMARY_DIR / "data_quality_summary.csv"
TRANSACTION_MISSING_FILE = SUMMARY_DIR / "transaction_missing_values.csv"
IDENTITY_MISSING_FILE = SUMMARY_DIR / "identity_missing_values.csv"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
train_transaction = pd.read_csv(TRAIN_TRANSACTION_FILE)
train_identity = pd.read_csv(TRAIN_IDENTITY_FILE)

train_transaction.head()
train_identity.head()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS


In [3]:
def calculate_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    missing_count = df.isna().sum()
    missing_percentage = df.isna().mean() * 100
    missing_df = pd.DataFrame(
        {
            "column_name": df.columns,
            "missing_count": missing_count.values,
            "missing_percentage": missing_percentage.values,
        }
    )
    return missing_df.sort_values("missing_percentage", ascending=False)

transaction_missing = calculate_missing_values(train_transaction)
identity_missing = calculate_missing_values(train_identity)

transaction_missing.head(15)
identity_missing.head(15)

,column_name,missing_count,missing_percentage
24,id_24,139486,96.708798
25,id_25,139101,96.441868
7,id_07,139078,96.425922
8,id_08,139078,96.425922
21,id_21,139074,96.423149
26,id_26,139070,96.420375
23,id_23,139064,96.416215
27,id_27,139064,96.416215
22,id_22,139064,96.416215
18,id_18,99120,68.722137


In [4]:
def plot_top_missing_values(missing_df: pd.DataFrame, title: str, output_file: Path, top_n: int = 20) -> None:
    top_missing = missing_df.head(top_n).sort_values("missing_percentage")
    plt.figure(figsize=(10, 7))
    plt.barh(top_missing["column_name"], top_missing["missing_percentage"])
    plt.title(title)
    plt.xlabel("Missing Percentage")
    plt.ylabel("Column")
    plt.tight_layout()
    plt.savefig(output_file)
    plt.close()

plot_top_missing_values(transaction_missing, "Top Missing Columns in train_transaction", FIGURES_DIR / "top_missing_transaction_columns.png")
plot_top_missing_values(identity_missing, "Top Missing Columns in train_identity", FIGURES_DIR / "top_missing_identity_columns.png")

In [5]:
def count_near_constant_columns(df: pd.DataFrame, threshold: float = 0.99) -> int:
    near_constant_count = 0
    for column in df.columns:
        top_frequency = df[column].value_counts(dropna=False, normalize=True).head(1)
        if not top_frequency.empty and top_frequency.iloc[0] >= threshold:
            near_constant_count += 1
    return near_constant_count

def get_high_cardinality_columns(df: pd.DataFrame, threshold: int = 100) -> list[str]:
    object_columns = df.select_dtypes(include=["object"]).columns
    high_cardinality_columns = []
    for column in object_columns:
        unique_count = df[column].nunique(dropna=True)
        if unique_count > threshold:
            high_cardinality_columns.append(column)
    return high_cardinality_columns

transaction_near_constant_count = count_near_constant_columns(train_transaction)
identity_near_constant_count = count_near_constant_columns(train_identity)
transaction_high_cardinality = get_high_cardinality_columns(train_transaction)
identity_high_cardinality = get_high_cardinality_columns(train_identity)

transaction_near_constant_count, identity_near_constant_count, transaction_high_cardinality, identity_high_cardinality

(15, 0, [], ['id_31', 'id_33', 'DeviceInfo'])

In [6]:
identity_ids = set(train_identity["TransactionID"])
transaction_ids = set(train_transaction["TransactionID"])
matched_identity_count = len(transaction_ids.intersection(identity_ids))
identity_coverage_percentage = (matched_identity_count / len(train_transaction)) * 100
fraud_percentage = train_transaction["isFraud"].mean() * 100
non_positive_amount_count = (train_transaction["TransactionAmt"] <= 0).sum()

amount_q1 = train_transaction["TransactionAmt"].quantile(0.25)
amount_q3 = train_transaction["TransactionAmt"].quantile(0.75)
amount_iqr = amount_q3 - amount_q1
amount_upper_bound = amount_q3 + (1.5 * amount_iqr)
amount_outlier_count = (train_transaction["TransactionAmt"] > amount_upper_bound).sum()

summary_records = [
    ("train_transaction_rows", train_transaction.shape[0]),
    ("train_transaction_columns", train_transaction.shape[1]),
    ("train_identity_rows", train_identity.shape[0]),
    ("train_identity_columns", train_identity.shape[1]),
    ("transaction_duplicate_transaction_ids", train_transaction["TransactionID"].duplicated().sum()),
    ("identity_duplicate_transaction_ids", train_identity["TransactionID"].duplicated().sum()),
    ("matched_identity_records", matched_identity_count),
    ("identity_coverage_percentage", round(identity_coverage_percentage, 3)),
    ("fraud_percentage", round(fraud_percentage, 3)),
    ("non_positive_transaction_amount_count", non_positive_amount_count),
    ("transaction_amount_iqr_upper_bound", round(amount_upper_bound, 3)),
    ("transaction_amount_outlier_count_iqr_rule", amount_outlier_count),
]

quality_summary = pd.DataFrame(summary_records, columns=["metric", "value"])
quality_summary

,metric,value
0,train_transaction_rows,590540.000
1,train_transaction_columns,394.000
2,train_identity_rows,144233.000
3,train_identity_columns,41.000
4,transaction_duplicate_transaction_ids,0.000
5,identity_duplicate_transaction_ids,0.000
6,matched_identity_records,144233.000
7,identity_coverage_percentage,24.424
8,fraud_percentage,3.499
9,non_positive_transaction_amount_count,0.000


In [7]:
quality_summary.to_csv(QUALITY_SUMMARY_FILE, index=False)
transaction_missing.to_csv(TRANSACTION_MISSING_FILE, index=False)
identity_missing.to_csv(IDENTITY_MISSING_FILE, index=False)

print(quality_summary)
print("\nTop 15 missing columns in train_transaction:")
print(transaction_missing.head(15))
print("\nTop 15 missing columns in train_identity:")
print(identity_missing.head(15))

                                       metric       value
0                      train_transaction_rows  590540.000
1                   train_transaction_columns     394.000
2                         train_identity_rows  144233.000
3                      train_identity_columns      41.000
4       transaction_duplicate_transaction_ids       0.000
5          identity_duplicate_transaction_ids       0.000
6                    matched_identity_records  144233.000
7                identity_coverage_percentage      24.424
8                            fraud_percentage       3.499
9       non_positive_transaction_amount_count       0.000
10         transaction_amount_iqr_upper_bound     247.519
11  transaction_amount_outlier_count_iqr_rule   66482.000

Top 15 missing columns in train_transaction:
    column_name  missing_count  missing_percentage
14        dist2         552913           93.628374
37           D7         551623           93.409930
43          D13         528588           89.509

## Key Findings

The dataset shows strong class imbalance, substantial missingness in several columns, partial identity coverage, and a long-tailed transaction amount distribution. These issues should be addressed during preprocessing and feature engineering before model training.